In [ ]:
# Simple LLM Workflow
# This notebook demonstrates integrating an LLM (ChatOpenAI) into a LangGraph workflow
# The workflow accepts a question and uses the LLM to generate an answer

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables from .env file
# This loads OPENAI_API_KEY and other configuration
load_dotenv()

False

In [ ]:
# Initialize the ChatOpenAI model
# Uses the OPENAI_API_KEY from environment variables
model = ChatOpenAI()

In [ ]:
# Define the state schema for the LLM workflow
class LLMState(TypedDict):
    """State for LLM Q&A workflow
    
    Attributes:
        question: The question to ask the LLM
        answer: The LLM's response (populated during workflow execution)
    """
    question: str
    answer: str

In [ ]:
# Define the workflow node that calls the LLM
def llm_qa(state: LLMState) -> LLMState:
    """Process a question through the LLM
    
    This node:
    1. Extracts the question from state
    2. Formats it into a prompt
    3. Sends it to the LLM (ChatOpenAI)
    4. Extracts the response content
    5. Updates the state with the answer
    
    Args:
        state: Current workflow state containing the question
        
    Returns:
        Updated state with the LLM's answer
    """
    question = state["question"]
    
    # Format the prompt with system instructions
    prompt = f"You are a helpful assistant. Please answer the following question: {question}"
    
    # Send the prompt to the LLM and get the response
    response = model.invoke(prompt)
    
    # Extract the text content from the response
    answer = response.content
    
    # Update the state with the answer
    state["answer"] = answer
    return state

In [ ]:
# Build and execute the LLM workflow
# Create the state graph
graph = StateGraph(LLMState)

# Add the LLM node to the graph
graph.add_node("ask_question", llm_qa)

# Define the workflow flow
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", END)

# Compile the workflow
workflow = graph.compile()

# Define the initial state with a question
initial_state = {
    "question": "Who is the president of INDIA",
    "answer": ""
}

# Execute the workflow
output_state = workflow.invoke(initial_state)

# Display the results
print("Question:")
print(output_state["question"])
print("\nAnswer:")
print(output_state["answer"])

{'question': 'Who is the president of INDIA', 'answer': 'The current President of India is Ram Nath Kovind.'}
